নিচের নোটবুকগুলি চালানোর জন্য, যদি এখনও না করে থাকেন, আপনাকে একটি মডেল ডিপ্লয় করতে হবে যা `text-embedding-ada-002` কে বেস মডেল হিসেবে ব্যবহার করে এবং ডিপ্লয়মেন্টের নাম .env ফাইলে `AZURE_OPENAI_EMBEDDINGS_ENDPOINT` হিসেবে সেট করতে হবে


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


পরবর্তীতে, আমরা এমবেডিং ইনডেক্সটি একটি Pandas ডেটাফ্রেমে লোড করব। এমবেডিং ইনডেক্সটি `embedding_index_3m.json` নামক একটি JSON ফাইলে সংরক্ষিত আছে। এমবেডিং ইনডেক্সে অক্টোবর ২০২৩ সালের শেষ পর্যন্ত প্রতিটি ইউটিউব ট্রান্সক্রিপ্টের এমবেডিংস রয়েছে।


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

পরবর্তী, আমরা `get_videos` নামে একটি ফাংশন তৈরি করতে যাচ্ছি যা কুয়েরির জন্য এম্বেডিং ইনডেক্স খুঁজবে। ফাংশনটি টপ ৫টি ভিডিও ফিরিয়ে দেবে যা কুয়েরির সাথে সবচেয়ে বেশি মিল রয়েছে। ফাংশনটি নিম্নরূপ কাজ করে:

১. প্রথমে, এম্বেডিং ইনডেক্সের একটি কপি তৈরি করা হয়।
২. পরবর্তীতে, OpenAI Embedding API ব্যবহার করে কুয়েরির জন্য এম্বেডিং হিসাব করা হয়।
৩. তারপর এম্বেডিং ইনডেক্সে `similarity` নামে একটি নতুন কলাম তৈরি করা হয়। `similarity` কলামে কুয়েরি এম্বেডিং এবং প্রতিটি ভিডিও সেগমেন্টের এম্বেডিংয়ের মধ্যে কোসাইন সাদৃশ্য থাকে।
৪. এরপর, এম্বেডিং ইনডেক্স `similarity` কলাম অনুযায়ী ফিল্টার করা হয়। এম্বেডিং ইনডেক্স শুধুমাত্র সেই ভিডিওগুলো অন্তর্ভুক্ত করার জন্য ফিল্টার করা হয় যেগুলোর কোসাইন সাদৃশ্য ০.৭৫ বা তার বেশি।
৫. শেষ পর্যন্ত, এম্বেডিং ইনডেক্স `similarity` কলাম অনুযায়ী সাজানো হয় এবং শীর্ষ ৫টি ভিডিও ফিরিয়ে দেওয়া হয়।


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

এই ফাংশনটি খুবই সহজ, এটি কেবল অনুসন্ধান ক্যোয়ারির ফলাফলগুলি মুদ্রণ করে। 


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

১. প্রথমে, এম্বেডিং ইনডেক্স একটি প্যান্ডাস ডেটাফ্রেমে লোড করা হয়।
২. এরপর, ব্যবহারকারীকে একটি প্রশ্ন লিখতে বলা হয়।
৩. তারপর `get_videos` ফাংশনটি কল করা হয় এম্বেডিং ইনডেক্সে প্রশ্ন অনুসন্ধানের জন্য।
৪. অবশেষে, `display_results` ফাংশনটি কল করা হয় ফলাফলগুলি ব্যবহারকারীকে দেখানোর জন্য।
৫. এরপর ব্যবহারকারীকে আরেকটি প্রশ্ন দিতে বলা হয়। এই প্রক্রিয়া চলতে থাকে যতক্ষণ না ব্যবহারকারী `exit` লিখে।

![](../../../../translated_images/bn/notebook-search.1e320b9c7fcbb0bc.webp)

আপনাকে একটি প্রশ্ন লিখতে বলা হবে। একটি প্রশ্ন লিখুন এবং এন্টার চাপুন। অ্যাপ্লিকেশনটি সেই প্রশ্নের সাথে সম্পর্কিত ভিডিওগুলোর একটি তালিকা প্রদর্শন করবে। অ্যাপ্লিকেশনটি প্রশ্নের উত্তরের অবস্থানসহ ভিডিওর লিঙ্কও প্রদান করবে।

এখানে কিছু প্রশ্ন দেওয়া হলো যা আপনি চেষ্টা করতে পারেন:

- Azure Machine Learning কী?
- Convolutional neural networks কীভাবে কাজ করে?
- Neural network কী?
- Azure Machine Learning-এর সাথে Jupyter Notebooks কি ব্যবহার করা যায়?
- ONNX কী?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
